# 📝 [제 9편] XOR 문제 - 직선 하나로는 나눌 수 없다

**학년/반/번호:** 2학년 ( )반 ( )번  **이름:** (           )

> 근거 자료: 박성호, 「머신러닝 XOR 문제 - AND • OR • NAND • XOR」 (ML_LEC_18), 「Python class」 (ML_LEC_05)

---

## 🎯 학습 목표
1. 논리게이트 AND, OR, NAND, XOR 의 진리표를 머신러닝 학습 데이터로 볼 수 있다.
2. 로지스틱 회귀 하나로 AND, OR, NAND 는 학습되지만 XOR 은 학습되지 않는 이유를 설명할 수 있다.
3. NAND, OR, AND 를 층으로 쌓아 XOR 을 해결하고, 이것이 신경망(딥러닝)의 핵심 아이디어임을 이해한다.

---

## 🧭 지난 시간 복습
* 제 8편: 로지스틱 회귀의 **결정 경계는 직선** (W₁x₁ + W₂x₂ + b = 0) 입니다.
* 도전 과제: (0,0)·(1,1) 은 0, (0,1)·(1,0) 은 1 인 점들을 **직선 하나로 나눌 수 있을까?**

[⬅ 이전 : 제 8편 로지스틱 회귀 - 합격일까 불합격일까? (분류)](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/08_logistic_regression.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/00_index.ipynb)　|　[다음 : 제 10편 신경망과 딥러닝 - 피드포워드로 XOR 풀기 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/10_neural_network.ipynb)

> 💡 **처음 여는 경우** : 파일 → **Drive에 사본 저장** 을 누르고, 내 사본에서 공부하세요. (빈칸을 채우고 저장할 수 있습니다)
>
> 🧹 **다 본 뒤에는** : 수정 → **모든 출력 지우기** 후 저장하면 애니메이션 때문에 무거워진 사본이 다시 가벼워집니다.

## ⚙️ 0단계. 준비하기
아래 셀을 **가장 먼저 한 번** 실행하세요. (그래프 한글 글꼴과 애니메이션 설정)

In [ ]:
#@title ▶ 준비 셀 (먼저 실행하세요)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import animation
from IPython.display import HTML, display
import urllib.request, os

font_path = 'NanumGothic.ttf'
try:
    if not os.path.exists(font_path):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf', font_path)
    fm.fontManager.addfont(font_path)
    plt.rc('font', family=fm.FontProperties(fname=font_path).get_name())
except Exception as e:
    print('글꼴을 내려받지 못했습니다(그래프의 한글이 깨질 수 있음):', e)
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['animation.embed_limit'] = 50
def show_anim(anim):
    # 애니메이션을 코랩 화면에 재생 버튼과 함께 보여 주기
    display(HTML(anim.to_jshtml()))
np.set_printoptions(precision=6, suppress=True)
print('✅ 준비 완료!')

In [ ]:
#@title ▶ 수치미분, sigmoid 함수 (실행만 하세요)
def numerical_derivative(f, x):
    delta_x = 1e-4
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + delta_x; fx1 = f(x)
        x[idx] = float(tmp_val) - delta_x; fx2 = f(x)
        grad[idx] = (fx1 - fx2) / (2 * delta_x)
        x[idx] = tmp_val
        it.iternext()
    return grad

def sigmoid(z):
    return 1 / (1 + np.exp(-z))
print('준비 완료')

## 📖 개념 다지기 1. 논리게이트 = 학습 데이터

컴퓨터 회로의 기본 부품인 논리게이트는 입력 2개 (0 또는 1) 를 받아 0 또는 1 을 출력합니다.
진리표를 **입력 (x₁, x₂), 정답 t** 로 보면 그대로 **분류 문제의 학습 데이터** 입니다.

| x₁ | x₂ | AND | OR | NAND | XOR |
|:-:|:-:|:-:|:-:|:-:|:-:|
| 0 | 0 | 0 | 0 | 1 | 0 |
| 0 | 1 | 0 | 1 | ( 　 ) | 1 |
| 1 | 0 | 0 | ( 　 ) | 1 | ( 　 ) |
| 1 | 1 | ( 　 ) | 1 | 0 | ( 　 ) |

* AND : 둘 다 1 일 때만 1
* OR : 하나라도 1 이면 1
* NAND : AND 의 반대 (Not AND)
* XOR : 두 입력이 **서로 다를 때만** 1 (eXclusive OR)

## 💻 실습 1. LogicGate 클래스 만들기

게이트 4개를 똑같은 방법으로 학습시키므로 **클래스(class)** 로 한 번 만들어 두고 여러 번 씁니다.

| 메서드 | 하는 일 |
|:---|:---|
| `__init__` | 게이트 이름, 입력/정답 데이터 저장, W(2 × 1), b 를 무작위로 초기화 |
| `__loss_func` | cross-entropy 손실 계산 (밖에서 못 쓰게 이름 앞에 `__`) |
| `error_val` | 손실 값 확인 |
| `train` | 수치미분 + 경사하강법으로 학습 |
| `predict` | sigmoid ≥ 0.5 이면 1, 아니면 0 |

In [ ]:
class LogicGate:
    def __init__(self, gate_name, xdata, tdata):
        self.name = gate_name
        self.__xdata = xdata.reshape(4, 2)          # 입력 (0,0), (0,1), (1,0), (1,1)
        self.__tdata = tdata.reshape(4, 1)          # 정답
        self.__W = np.random.rand(2, 1)             # 가중치 W1, W2
        self.__b = np.random.rand(1)                # 바이어스
        self.__learning_rate = 1e-2
        self.history = []                           # 애니메이션용 기록

    def __loss_func(self):                          # cross-entropy
        delta = 1e-7
        z = np.dot(self.__xdata, self.__W) + self.__b
        y = sigmoid(z)
        return -np.sum(self.__tdata*np.log(y + delta) + (1 - self.__tdata)*np.log((1 - y) + delta))

    def error_val(self):
        return self.__loss_func()

    def train(self, steps=8001, show=True):
        f = lambda x : self.__loss_func()
        if show: print(f'[{self.name}] 처음 손실 =', round(self.error_val(), 5))
        for step in range(steps):
            self.__W -= self.__learning_rate * numerical_derivative(f, self.__W)
            self.__b -= self.__learning_rate * numerical_derivative(f, self.__b)
            if step % 100 == 0:
                self.history.append((step, self.__W.copy(), self.__b.copy()))
            if show and step % 2000 == 0:
                print(f'   step {step:5d} : 손실 = {self.error_val():.5f}')

    def predict(self, input_data):
        z = np.dot(input_data, self.__W) + self.__b
        y = sigmoid(z)
        result = 1 if y[0] > 0.5 else 0
        return y[0], result

print('LogicGate 클래스 준비 완료')

### AND, OR, NAND 학습시키기

In [ ]:
np.random.seed(0)
xdata = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
gates = {}
for name, tdata in [('AND', [0, 0, 0, 1]), ('OR', [0, 1, 1, 1]), ('NAND', [1, 1, 1, 0])]:
    g = LogicGate(name + '_GATE', xdata, np.array(tdata))
    g.train()
    gates[name] = g

print()
for name, g in gates.items():
    print(f'{name:5s}:', end=' ')
    for x in xdata:
        y, r = g.predict(x)
        print(f'{x.tolist()} → {r}', end='   ')
    print()

> **[확인 1]** AND, OR, NAND 의 예측 결과가 진리표와 모두 일치하나요? ( 예 / 아니오 )

### 이제 XOR 차례!

In [ ]:
np.random.seed(0)
xor = LogicGate('XOR_GATE', xdata, np.array([0, 1, 1, 0]))
xor.train()
gates['XOR'] = xor
for x in xdata:
    y, r = xor.predict(x)
    print(f'{x.tolist()} → sigmoid = {y:.4f} → 예측 {r}')

> **[확인 2]** XOR 의 예측 결과는 정답 (0, 1, 1, 0) 과 ( 일치한다 / 다르다 ).
> 손실 값은 학습을 8000 번 해도 약 ( 　　 ) 에서 더 이상 줄지 않는다. sigmoid 값들은 모두 ( 　　 ) 근처이다.

## 🎬 애니메이션 1. 네 게이트의 결정 경계가 학습되는 모습

배경색은 출력이 1 일 확률(빨강 1, 파랑 0), 검은 선은 결정 경계(sigmoid = 0.5) 입니다. ● 정답 1, ✕ 정답 0

In [ ]:
#@title 🎬 애니메이션 1 실행 (실습 1 을 먼저 실행하세요)
G1, G2 = np.meshgrid(np.linspace(-0.5, 1.5, 60), np.linspace(-0.5, 1.5, 60))
grid = np.c_[G1.ravel(), G2.ravel()]
targets = {'AND': [0, 0, 0, 1], 'OR': [0, 1, 1, 1], 'NAND': [1, 1, 1, 0], 'XOR': [0, 1, 1, 0]}
n = len(gates['AND'].history)
sel = list(range(0, 20)) + list(range(20, n, 3)) + [n - 1]*6
fig, axes = plt.subplots(1, 4, figsize=(16, 4.3))
def draw(f):
    k = sel[f]
    for ax, name in zip(axes, ['AND', 'OR', 'NAND', 'XOR']):
        st, Wk, bk = gates[name].history[k]
        P = sigmoid(grid @ Wk + bk).reshape(G1.shape)
        ax.clear()
        ax.contourf(G1, G2, P, levels=np.linspace(0, 1, 11), cmap='coolwarm', alpha=0.6)
        if P.min() < 0.5 < P.max():
            ax.contour(G1, G2, P, levels=[0.5], colors='k', linewidths=2.5)
        for (a, b2), tt in zip(xdata, targets[name]):
            ax.scatter(a, b2, s=160, marker='o' if tt else 'x', color='darkred' if tt else 'navy', lw=3, zorder=5)
        ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5); ax.set_aspect('equal')
        ax.set_title(f'{name}   (step {st})', fontsize=13)
anim = animation.FuncAnimation(fig, draw, frames=len(sel), interval=120)
plt.close(fig)
show_anim(anim)

> **[관찰 1]** AND, OR, NAND 는 직선 하나로 ● 와 ✕ 가 깔끔하게 나뉜다. XOR 은 배경이 거의 ( 한 가지 색 / 두 가지 색 ) 이고, ● 와 ✕ 를 나누지 못한다.

## 🎬 애니메이션 2. 정말로 직선 하나로는 안 될까?

직선을 여러 방향, 여러 위치로 움직여 보며 XOR 의 네 점 중 **몇 개를 맞히는지** 세어 봅시다.

In [ ]:
#@title 🎬 애니메이션 2 실행 (직선 하나로 XOR 나누기 시도)
pts = np.array([[0, 0], [0, 1], [1, 0], [1, 1]]); tx = np.array([0, 1, 1, 0])
lines = []                                   # (법선 방향 각도, 중심 (0.5, 0.5) 에서 떨어진 거리)
for ang in np.linspace(0, np.pi, 36, endpoint=False):
    for c in [-0.35, 0.0, 0.35]:
        lines.append((ang, c))
fig, ax = plt.subplots(figsize=(6, 6))
best = [0]
center = np.array([0.5, 0.5])
def draw(i):
    ang, c = lines[i]
    nvec = np.array([np.cos(ang), np.sin(ang)])        # 직선에 수직인 방향
    p0 = center + c*nvec                               # 직선 위의 한 점
    side = ((pts - p0) @ nvec > 0).astype(int)         # 직선의 어느 쪽에 있는가
    correct = max((side == tx).sum(), ((1 - side) == tx).sum())
    best[0] = max(best[0], correct)
    ax.clear()
    t_ = np.linspace(-3, 3, 10)
    d = np.array([-nvec[1], nvec[0]])
    ax.plot(p0[0] + t_*d[0], p0[1] + t_*d[1], 'g-', lw=3)
    for (a, b2), tt in zip(pts, tx):
        ax.scatter(a, b2, s=200, marker='o' if tt else 'x', color='darkred' if tt else 'navy', lw=3, zorder=5)
    ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5); ax.set_aspect('equal'); ax.grid(alpha=0.3)
    ax.set_title(f'이 직선으로 맞힌 점 : {correct} / 4    (지금까지 최고 {best[0]} / 4)', fontsize=12)
anim = animation.FuncAnimation(fig, draw, frames=len(lines), interval=100)
plt.close(fig)
show_anim(anim)

> **[관찰 2]** 어떤 직선을 그어도 최대 ( 　 ) / 4 개밖에 맞히지 못한다.
> 💡 이처럼 직선 하나로 나눌 수 없는 데이터를 **선형 분리 불가능** 하다고 합니다. 1969년 이 사실이 알려지면서 인공지능 연구가 한동안 침체되기도 했습니다.

## 📖 개념 다지기 2. 게이트를 쌓아서 XOR 만들기

XOR 은 NAND 와 OR 의 출력을 다시 AND 에 넣으면 만들 수 있습니다.

```
 x₁ ─┬─→ [NAND] ─ s₁ ─┐
     │                 ├─→ [AND] ─→ y (XOR)
 x₂ ─┴─→ [ OR ] ─ s₂ ─┘
```

**[연습 1]** 표를 완성하세요.

| x₁ | x₂ | s₁ = NAND(x₁, x₂) | s₂ = OR(x₁, x₂) | y = AND(s₁, s₂) | XOR 정답 |
|:-:|:-:|:-:|:-:|:-:|:-:|
| 0 | 0 | 1 | 0 | ( 　 ) | 0 |
| 0 | 1 | ( 　 ) | 1 | ( 　 ) | 1 |
| 1 | 0 | 1 | ( 　 ) | 1 | 1 |
| 1 | 1 | ( 　 ) | ( 　 ) | ( 　 ) | 0 |

In [ ]:
# 실습 1 에서 학습시킨 NAND, OR, AND 게이트를 조합
for x in xdata:
    _, s1 = gates['NAND'].predict(x)          # 1층
    _, s2 = gates['OR'].predict(x)            # 1층
    _, y  = gates['AND'].predict(np.array([s1, s2]))   # 2층 : 1층의 출력이 2층의 입력
    print(f'{x.tolist()} → s1 = {s1}, s2 = {s2} → XOR = {y}')

## 🎬 애니메이션 3. 1층이 '공간을 바꿔' 준다

1층(NAND, OR)을 지나면 점 (x₁, x₂) 가 새로운 점 (s₁, s₂) 로 **옮겨집니다.**
옮겨진 공간에서는 **직선 하나(AND 의 결정 경계)로 나눌 수 있게** 됩니다!

In [ ]:
#@title 🎬 애니메이션 3 실행 (공간 변환)
P0 = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
P1 = np.array([[1, 0], [1, 1], [1, 1], [0, 1]], dtype=float)        # (NAND, OR)
tx = [0, 1, 1, 0]; labels = ['(0,0)', '(0,1)', '(1,0)', '(1,1)']
offs = np.array([[-0.12, -0.12], [-0.12, 0.08], [0.06, -0.12], [0.06, 0.08]])
ts = np.concatenate([np.zeros(12), np.linspace(0, 1, 30), np.ones(25)])
fig, ax = plt.subplots(figsize=(6.5, 6.5))
def draw(i):
    a = ts[i]; Pc = (1 - a)*P0 + a*P1
    ax.clear()
    for k in range(4):
        jitter = np.array([0.05, -0.05])*(k == 1) + np.array([-0.05, 0.05])*(k == 2) if a == 1 else 0
        ax.scatter(*(Pc[k] + jitter), s=220, marker='o' if tx[k] else 'x', color='darkred' if tx[k] else 'navy', lw=3, zorder=5)
        ax.text(*(Pc[k] + offs[k]), labels[k], fontsize=11)
    if a == 1:
        xs = np.linspace(-0.5, 1.5, 10)
        ax.plot(xs, 1.5 - xs, 'g-', lw=3, label='AND 의 결정 경계 s1 + s2 = 1.5')
        ax.legend(loc='lower left')
        ax.set_title('2단계 : (s1, s2) 공간에서는 직선으로 나뉜다!', fontsize=13)
        ax.set_xlabel('s1 = NAND'); ax.set_ylabel('s2 = OR')
    else:
        ax.set_title('1단계 : (x1, x2) → (NAND, OR) 로 이동 중', fontsize=13)
        ax.set_xlabel('x1 → s1'); ax.set_ylabel('x2 → s2')
    ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5); ax.set_aspect('equal'); ax.grid(alpha=0.3)
anim = animation.FuncAnimation(fig, draw, frames=len(ts), interval=90)
plt.close(fig)
show_anim(anim)

> **[관찰 3]** (0, 1) 과 (1, 0) 은 이동한 뒤 같은 점 ( 　, 　 ) 으로 모인다. (0, 0) 은 ( 　, 　 ), (1, 1) 은 ( 　, 　 ) 으로 옮겨진다.

## 💡 insight - 신경망(딥러닝)의 핵심 아이디어

* 게이트 하나하나는 **로지스틱 회귀** (선형 회귀 + sigmoid) 입니다.
* 앞 층의 **출력**을 다음 층의 **입력**으로 넣어 **여러 층(multi-layer)** 으로 쌓으면, 직선 하나로는 풀 수 없던 문제도 풀 수 있습니다.
* 각 층은 데이터를 **나누기 쉬운 공간으로 옮겨 주는** 역할을 합니다.

→ 이것이 바로 **신경망(Neural Network)** 이고, 층을 깊게(deep) 쌓은 것이 **딥러닝(Deep Learning)** 입니다!

## ✏️ 연습 문제

> **[문제 1]** XNOR 은 XOR 의 반대 (두 입력이 같을 때 1) 입니다. XNOR 의 진리표를 쓰고, 로지스틱 회귀 하나로 학습이 될지 예상하세요.
> 진리표: (0,0)→( 　 ), (0,1)→( 　 ), (1,0)→( 　 ), (1,1)→( 　 ),  예상: ( 된다 / 안 된다 )

> **[문제 2]** XOR = AND(NAND, OR) 이었습니다. 그렇다면 XNOR 은 어떤 게이트들의 조합으로 만들 수 있을까요? 아래 코드의 `______` 에 게이트 이름을 넣어 확인하세요. (힌트: XNOR = NOT XOR, 즉 마지막을 AND 대신 ( 　　 ) 로!)

In [ ]:
for x in xdata:
    _, s1 = gates['NAND'].predict(x)
    _, s2 = gates['OR'].predict(x)
    _, y  = gates['______'].predict(np.array([s1, s2]))
    print(f'{x.tolist()} → XNOR = {y}')

In [ ]:
#@title 🔑 정답 확인 { display-mode: "form" }
print('문제 1 : XNOR 진리표 (0,0)→1, (0,1)→0, (1,0)→0, (1,1)→1. XOR 처럼 직선 하나로 나눌 수 없으므로 학습이 안 된다.')
print('문제 2 : 빈칸 = NAND  →  XNOR = NAND(NAND, OR)')
for x in xdata:
    _, s1 = gates['NAND'].predict(x); _, s2 = gates['OR'].predict(x)
    _, y = gates['NAND'].predict(np.array([s1, s2]))
    print('   ', x.tolist(), '→', y)

## 📌 정리

| 핵심 | 내용 |
|:---|:---|
| 논리게이트 | 진리표 = 입력 (x₁, x₂) 와 정답 t 로 된 분류 학습 데이터 |
| AND, OR, NAND | 직선 하나로 나뉜다 → 로지스틱 회귀 하나로 학습 성공 |
| XOR | 직선 하나로 나뉘지 않는다 → 로지스틱 회귀 하나로는 실패 |
| 해결 | NAND, OR (1층) → AND (2층) 처럼 **층을 쌓는다** = 신경망의 아이디어 |

### 🏆 도전 과제
이번에는 사람이 "NAND, OR, AND 를 쌓자" 고 **설계**해 주었습니다. 만약 층의 가중치를 모두 처음부터 **스스로 학습**하게 하면 어떻게 될까요?
→ 다음 시간: **신경망과 딥러닝**  (                                                                         )

---

[⬅ 이전 : 제 8편 로지스틱 회귀 - 합격일까 불합격일까? (분류)](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/08_logistic_regression.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/00_index.ipynb)　|　[다음 : 제 10편 신경망과 딥러닝 - 피드포워드로 XOR 풀기 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/10_neural_network.ipynb)